# آماده‌سازی مرتب BUSI
نسخهٔ بازسازی مستقل: فایل‌های اصلی تغییر نمی‌کنند؛ گزارش جدید در پوشهٔ جدا ذخیره می‌شود. برای آموزش لازم نیست این نوت‌بوک دوباره اجرا شود؛ تقسیم قبلی ذخیره شده است.

اگر قصد بازسازی دارید، سلول‌ها را از بالا به پایین اجرا کنید. GPU برای آماده‌سازی لازم نیست. تصمیم‌های بررسی بصری قبلی صریح ثبت شده‌اند؛ اجرای کد بررسی بصری جدید نیست. گروه شباهت، شناسهٔ بیمار نیست.

## ۱. اتصال Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ۲. کتابخانه‌ها و مسیرها
ROOT مسیر تصاویر است. REFERENCE_SPLIT تقسیم تأییدشدهٔ قبلی است؛ در پایان با نتیجهٔ جدید مقایسه می‌شود.

In [ ]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path("/content/drive/MyDrive/BUSI_MERGED")
CLASSES = ('normal', 'benign', 'malignant')
EXPECTED = {'normal': 133, 'benign': 437, 'malignant': 210}
REFERENCE_SPLIT = Path("/content/drive/MyDrive/BUSI_AUDIT_REPORTS/revision_20261004T122828Z/split_manifest.csv")
assert ROOT.is_dir(), f'Dataset not found: {ROOT}'
reference_split = pd.read_csv(REFERENCE_SPLIT) if REFERENCE_SPLIT.is_file() else None
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
report_dir_v2 = ROOT.parent / 'BUSI_AUDIT_REPORTS' / f'clean_rebuild_{stamp}'
report_dir_v2.mkdir(parents=True, exist_ok=False)
print('Dataset:', ROOT)
print('New reports:', report_dir_v2)


## ۳. بررسی داده
تعداد، خوانایی، اندازه، ماسک و تکرار دقیق بررسی می‌شود. تعارض شناخته‌شده در مرحلهٔ بعد کنار گذاشته می‌شود. این بخش ممکن است روی Drive چند دقیقه زمان ببرد.

In [ ]:
def audit_dataset(root):
    rows = []
    issues = []
    counts = []

    # ثبت مشکلات برای بررسی؛ هیچ فایلی تغییر نمی‌کند.
    def issue(severity, cls, file, reason):
        issues.append({
            'severity': severity,
            'class_name': cls,
            'file': str(file),
            'reason': reason
        })

    extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

    for cls in CLASSES:
        print(f'\nChecking class: {cls}', flush=True)

        # نام واقعی پوشه‌های دیتاست تو
        image_dir = root / cls / 'images'
        mask_dir = root / cls / 'masks'

        if not image_dir.is_dir() or not mask_dir.is_dir():
            issue(
                'ERROR', cls, root / cls,
                'Missing images/ or masks/ folder'
            )
            counts.append({
                'class_name': cls,
                'images': 0,
                'masks': 0,
                'expected': EXPECTED[cls]
            })
            continue

        images = sorted(
            p for p in image_dir.iterdir()
            if p.is_file() and p.suffix.lower() in extensions
        )

        masks = sorted(
            p for p in mask_dir.iterdir()
            if p.is_file() and p.suffix.lower() in extensions
        )

        print(
            f'Found {len(images)} images and {len(masks)} masks',
            flush=True
        )

        counts.append({
            'class_name': cls,
            'images': len(images),
            'masks': len(masks),
            'expected': EXPECTED[cls]
        })

        if len(images) != EXPECTED[cls]:
            issue(
                'WARNING', cls, image_dir,
                'Count differs from original BUSI; investigate preprocessing history'
            )

        used_masks = set()

        for index, ip in enumerate(images):
            # گزارش پیشرفت در اولین تصویر و سپس هر ۲۵ تصویر
            if index % 25 == 0:
                print(
                    f'  Reading {index + 1}/{len(images)}: {ip.name}',
                    flush=True
                )

            if '_mask' in ip.stem.lower():
                issue(
                    'ERROR', cls, ip,
                    'Mask-like filename inside images folder'
                )

            # مثال: benign (1).png → benign (1)_mask.png
            mp = mask_dir / (ip.stem + '_mask.png')

            row = {
                'class_name': cls,
                'image': str(ip.relative_to(root)),
                'mask': str(mp.relative_to(root)),
                'width': None,
                'height': None,
                'image_hash': None,
                'mask_values': None,
                'foreground_fraction': None,
                'pair_ok': False
            }

            # خواندن تصویر و محاسبه شناسه پیکسل‌های آن
            try:
                with Image.open(ip) as im:
                    rgb = np.array(im.convert('RGB'))

                row['height'], row['width'] = rgb.shape[:2]

                row['image_hash'] = hashlib.sha256(
                    str(rgb.shape).encode() + rgb.tobytes()
                ).hexdigest()

            except Exception as exc:
                issue(
                    'ERROR', cls, ip,
                    f'Cannot decode image: {exc}'
                )
                rows.append(row)
                continue

            # بررسی وجود ماسک متناظر
            if not mp.is_file():
                issue(
                    'ERROR', cls, ip,
                    'Matching *_mask.png is missing'
                )
                rows.append(row)
                continue

            used_masks.add(mp)

            try:
                with Image.open(mp) as mask_im:
                    raw = np.array(mask_im)
                    mask = np.array(mask_im.convert('L'))

                # ماسک رنگی با کانال‌های متفاوت نیاز به بررسی دارد.
                if raw.ndim == 3 and raw.shape[2] >= 3:
                    channels_equal = (
                        np.array_equal(raw[..., 0], raw[..., 1])
                        and np.array_equal(raw[..., 1], raw[..., 2])
                    )

                    if not channels_equal:
                        issue(
                            'ERROR', cls, mp,
                            'Mask has unequal RGB channels; inspect color-label encoding'
                        )

                values = np.unique(mask)
                row['mask_values'] = json.dumps(values.tolist())

                # سهم پیکسل‌های غیرصفر؛ فقط برای بررسی اولیه
                row['foreground_fraction'] = float((mask > 0).mean())

                same_size = mask.shape == rgb.shape[:2]

                if not same_size:
                    issue(
                        'ERROR', cls, mp,
                        'Image and mask sizes differ'
                    )

                binary_values = (
                    set(values.tolist()) <= {0, 1}
                    or set(values.tolist()) <= {0, 255}
                )

                if not binary_values:
                    issue(
                        'WARNING', cls, mp,
                        'Nonbinary mask values; inspect before deciding a threshold'
                    )

                if cls == 'normal' and np.any(mask > 0):
                    issue(
                        'ERROR', cls, mp,
                        'Normal mask is not empty'
                    )

                if cls != 'normal' and not np.any(mask > 0):
                    issue(
                        'ERROR', cls, mp,
                        'Lesion class has an empty mask'
                    )

                # فقط یعنی تصویر و ماسک خوانده شدند و اندازه یکسان دارند.
                # صحت کامل ماسک را باید از جدول مشکلات و نمایش تصویری بررسی کرد.
                row['pair_ok'] = same_size

            except Exception as exc:
                issue(
                    'ERROR', cls, mp,
                    f'Cannot decode mask: {exc}'
                )

            rows.append(row)

        # ماسک‌هایی که هیچ تصویر متناظری ندارند
        for mp in masks:
            if mp not in used_masks:
                issue(
                    'ERROR', cls, mp,
                    'Orphan / extra mask without a matched image'
                )

        print(f'Finished class: {cls}', flush=True)

    print('\nChecking exact duplicate images...', flush=True)

    groups = defaultdict(list)

    for row in rows:
        if row['image_hash']:
            groups[row['image_hash']].append(row)

    duplicate_rows = []
    group_number = 0

    for digest, members in sorted(groups.items()):
        if len(members) < 2:
            continue

        group_number += 1

        # یک تصویر یکسان با برچسب‌های متفاوت، تعارض برچسب دارد.
        conflict = len({r['class_name'] for r in members}) > 1

        for r in members:
            duplicate_rows.append({
                'group_id': group_number,
                'image_hash': digest,
                'class_name': r['class_name'],
                'image': r['image'],
                'cross_class_conflict': conflict
            })

            issue(
                'ERROR' if conflict else 'WARNING',
                r['class_name'],
                r['image'],
                (
                    'Exact image duplicate across classes'
                    if conflict
                    else 'Exact image duplicate; keep group in a single split later'
                )
            )

    return (
        pd.DataFrame(rows, columns=[
            'class_name', 'image', 'mask', 'width', 'height',
            'image_hash', 'mask_values',
            'foreground_fraction', 'pair_ok'
        ]),
        pd.DataFrame(issues, columns=[
            'severity', 'class_name', 'file', 'reason'
        ]),
        pd.DataFrame(counts, columns=[
            'class_name', 'images', 'masks', 'expected'
        ]),
        pd.DataFrame(duplicate_rows, columns=[
            'group_id', 'image_hash', 'class_name',
            'image', 'cross_class_conflict'
        ])
    )


# اجرای تابع و دریافت چهار جدول
manifest, issues, counts, duplicates = audit_dataset(ROOT)

print('\nCLASS COUNTS')
display(counts)

errors = int((issues['severity'] == 'ERROR').sum())
warnings = int((issues['severity'] == 'WARNING').sum())

summary = {
    'total_images': len(manifest),
    'errors': errors,
    'warnings': warnings,
    'exact_duplicate_groups': int(duplicates['group_id'].nunique()),
    'root': str(ROOT),
    'checked_at_utc': datetime.now(timezone.utc).isoformat()
}

print(json.dumps(summary, indent=2))

print('ERROR: investigate before training.')
print('WARNING: review; it does not necessarily mean the file is unusable.')
print('This audit does not rule out near-duplicates or patient leakage.')

if not issues.empty:
    print('\nISSUES — first 40 rows')
    display(issues.head(40))

if not duplicates.empty:
    print('\nEXACT DUPLICATES — first 30 rows')
    display(duplicates.head(30))
assert len(manifest) == 780
assert manifest['image'].is_unique
assert manifest['pair_ok'].all(), 'Resolve image/mask errors.'
assert manifest.groupby('class_name').size().to_dict() == EXPECTED
unresolved = issues[(issues['severity'] == 'ERROR') & (issues['reason'] != 'Exact image duplicate across classes')]
assert unresolved.empty, unresolved.to_string(index=False)
for filename, table in [('manifest',manifest),('issues',issues),('class_counts',counts),('exact_duplicates',duplicates)]:
    table.to_csv(report_dir_v2 / f'{filename}.csv', index=False)
(report_dir_v2 / 'audit_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')


## ۴. ثبت کنارگذاری ۱۹ تصویر
گروه‌های تعارض از گزارش علمی آمده‌اند و شباهت آن‌ها قبلاً توسط شما بررسی شده است. برچسب جدید تعیین نمی‌کنیم؛ فایل‌های اصلی نگه داشته می‌شوند.

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path("/content/drive/MyDrive/BUSI_MERGED")

# هر گروه شامل تصاویر مشابه با برچسب‌های متفاوت است.
conflict_groups = [
    [("benign", 42), ("benign", 131), ("malignant", 51)],
    [("benign", 85), ("benign", 164), ("malignant", 52)],
    [("benign", 333), ("malignant", 77), ("malignant", 78)],
    [("benign", 399), ("malignant", 93)],
    [("benign", 406), ("malignant", 94)],
    [("benign", 433), ("malignant", 145)],
    [("benign", 269), ("normal", 1)],
    [("benign", 437), ("normal", 34)],
]

from pathlib import Path
from datetime import datetime, timezone

# باید از فهرست اولیهٔ ۷۸۰ تصویر شروع کنیم.
assert len(manifest) == 780, (
    "manifest باید فهرست اولیهٔ ۷۸۰ تصویر باشد."
)
assert manifest["image"].is_unique, "مسیر تصویر تکراری است."

# نام فایل‌های ۱۹ تصویر، از گروه‌هایی که بررسی کردی
conflict_images = {
    f"{class_name}/images/{class_name} ({number}).png"
    for group in conflict_groups
    for class_name, number in group
}

assert len(conflict_images) == 19

# True یعنی این تصویر جزو موارد تعارض است.
is_conflict = manifest["image"].isin(conflict_images)

# فهرست کنارگذاشته‌شده‌ها، همراه دلیل
excluded_manifest_v2 = manifest.loc[is_conflict].copy()
excluded_manifest_v2["reason"] = (
    "Published BUSI cross-class duplicate; "
    "visual similarity confirmed by user; label unresolved"
)

# فهرست تصاویر باقی‌مانده
clean_manifest_v2 = (
    manifest.loc[~is_conflict]
    .copy()
    .reset_index(drop=True)
)

assert len(excluded_manifest_v2) == 19
assert len(clean_manifest_v2) == 761

clean_manifest_v2.to_csv(
    report_dir_v2 / "clean_manifest.csv", index=False
)
excluded_manifest_v2.to_csv(
    report_dir_v2 / "excluded_manifest.csv", index=False
)

print("Original images:", len(manifest))
print("Excluded images:", len(excluded_manifest_v2))
print("Remaining images:", len(clean_manifest_v2))

display(
    clean_manifest_v2.groupby("class_name")
    .size()
    .rename("images")
    .reset_index()
)

print("Reports saved:", report_dir_v2)

## ۵. نمایش نمونه‌های تصویر و ماسک — اختیاری
قرمز، ماسک مرجع است؛ پیش‌بینی مدل نیست.

In [ ]:
rng = np.random.default_rng(42)
for cls in CLASSES:
    valid = clean_manifest_v2[
    (clean_manifest_v2['class_name'] == cls)
    & clean_manifest_v2['pair_ok']
]
    if valid.empty:
        print('No readable matched pairs for:', cls)
        continue
    chosen = valid.iloc[rng.choice(len(valid), size=min(3, len(valid)), replace=False)]
    fig, axes = plt.subplots(len(chosen), 3, figsize=(12, 3.5 * len(chosen)), squeeze=False)
    for idx, (_, row) in enumerate(chosen.iterrows()):
        with Image.open(ROOT / row['image']) as im:
            img = np.array(im.convert('RGB'))
        with Image.open(ROOT / row['mask']) as im:
            mask = np.array(im.convert('L')) > 0
        axes[idx,0].imshow(img)
        axes[idx,0].set_title(row['image'], fontsize=9)
        axes[idx,1].imshow(mask, cmap='gray', vmin=0, vmax=1)
        axes[idx,1].set_title('Mask: foreground > 0 (audit display only)')
        axes[idx,2].imshow(img)
        overlay = np.zeros((*mask.shape, 4))
        overlay[mask] = (1, 0, 0, 0.4)
        axes[idx,2].imshow(overlay)
        axes[idx,2].set_title('Overlay')
        for ax in axes[idx]: ax.axis('off')
    fig.suptitle(cls)
    plt.tight_layout()
    plt.show()

## ۶. فهرست ثابت ۳۵ زوج تأییدشده
نام زوج‌های بررسی‌شده را ثبت می‌کنیم تا متغیر START و اجرای چندبارهٔ سلول‌ها لازم نباشد.

In [ ]:
confirmed_cases = [
    ('normal', 97, 98),
    ('benign', 322, 5),
    ('normal', 45, 62),
    ('benign', 12, 329),
    ('benign', 235, 294),
    ('benign', 249, 282),
    ('benign', 232, 276),
    ('benign', 128, 30),
    ('benign', 226, 263),
    ('benign', 255, 277),
    ('benign', 273, 289),
    ('benign', 238, 300),
    ('benign', 232, 287),
    ('malignant', 116, 18),
    ('normal', 58, 69),
    ('benign', 276, 287),
    ('benign', 274, 290),
    ('benign', 225, 292),
    ('benign', 139, 65),
    ('benign', 139, 157),
    ('benign', 312, 421),
    ('normal', 58, 60),
    ('normal', 41, 47),
    ('benign', 217, 268),
    ('benign', 163, 86),
    ('benign', 152, 18),
    ('benign', 218, 296),
    ('benign', 233, 299),
    ('normal', 131, 19),
    ('normal', 129, 35),
    ('normal', 51, 54),
    ('benign', 266, 279),
    ('benign', 263, 278),
    ('benign', 242, 288),
    ('benign', 3, 320),
]
reviewed_pairs = pd.DataFrame([
    {'image_1': f'{cls}/images/{cls} ({a}).png',
     'image_2': f'{cls}/images/{cls} ({b}).png',
     'decision': 'keep_in_same_split',
     'review_basis': 'previous_user_visual_review'}
    for cls, a, b in confirmed_cases
])
assert len(reviewed_pairs) == 35
original_paths = set(manifest['image'])
assert set(reviewed_pairs['image_1']).issubset(original_paths)
assert set(reviewed_pairs['image_2']).issubset(original_paths)
reviewed_pairs.to_csv(report_dir_v2 / 'reviewed_pairs.csv', index=False)
print('Previously reviewed pairs:', len(reviewed_pairs))


## ۷. استخراج جدول علمی
منبع: https://www.ippt.pan.pl/repository/open/o8361.pdf
DOI: 10.1016/j.dib.2023.109247
شماره‌های پیوستهٔ گزارش به مسیر فایل تبدیل می‌شوند. در صورت شکست دانلود، PDF ذخیره‌شدهٔ قبلی استفاده می‌شود.

In [ ]:
!pip -q install pypdf

In [ ]:
import requests
import re
import pandas as pd
from pypdf import PdfReader

# دانلود گزارش علمی
url = "https://www.ippt.pan.pl/repository/open/o8361.pdf"
pdf_path = report_dir_v2 / 'BUSI_published_review.pdf'
try:
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    assert response.content.startswith(b'%PDF'), 'Response is not a PDF.'
    pdf_path.write_bytes(response.content)
except (requests.RequestException, AssertionError):
    cached_pdf = REFERENCE_SPLIT.parent / 'BUSI_published_review.pdf'
    if not cached_pdf.is_file():
        raise RuntimeError('Download failed; cached report PDF is unavailable.')
    pdf_path.write_bytes(cached_pdf.read_bytes())
    print('Using previously saved PDF.')

# استخراج متن صفحات PDF
reader = PdfReader(str(pdf_path))
text = "\n".join(
    page.extract_text() or ""
    for page in reader.pages
)

# جدول B1 شامل زوج‌های تکراری است.
# آخرین محل عنوان را می‌گیریم تا ارجاع داخل متن انتخاب نشود.
start = text.rfind("Table B1")
end = text.rfind("Table B2")

assert 0 <= start < end, "عنوان جدول‌ها پیدا نشد."

table_text = text[start:end]

# استخراج زوج‌هایی مثل {1, 318}
number_pairs = re.findall(
    r"\{\s*(\d+)\s*,\s*(\d+)\s*\}",
    table_text
)

number_pairs = sorted({
    (int(a), int(b))
    for a, b in number_pairs
})

assert len(number_pairs) > 100, (
    "استخراج جدول ناقص به نظر می‌رسد؛ ادامه نده."
)
assert all(
    1 <= a <= 780 and 1 <= b <= 780
    for a, b in number_pairs
)

# گزارش شماره‌های هر سه کلاس را پشت سر هم نوشته است.
# این تابع شمارهٔ گزارش را به مسیر واقعی فایل تبدیل می‌کند.
def report_id_to_path(number):
    if number <= 437:
        class_name = "benign"
        file_number = number
    elif number <= 647:
        class_name = "malignant"
        file_number = number - 437
    else:
        class_name = "normal"
        file_number = number - 647

    return (
        f"{class_name}/images/"
        f"{class_name} ({file_number}).png"
    )

published_pairs = pd.DataFrame([
    {
        "image_1": report_id_to_path(a),
        "image_2": report_id_to_path(b)
    }
    for a, b in number_pairs
])

# اطمینان از اینکه مسیرها در فهرست اولیه وجود دارند
original_paths = set(manifest["image"])

assert set(published_pairs["image_1"]).issubset(original_paths)
assert set(published_pairs["image_2"]).issubset(original_paths)

published_pairs.to_csv(
    report_dir_v2 / "published_duplicate_pairs.csv",
    index=False
)

print("Extracted pairs:", len(published_pairs))
display(published_pairs.head(10))
assert len(published_pairs) == 226, 'Check PDF extraction: expected 226 pairs.'


## ۸. گروه‌بندی کامل
روابط گزارش، بررسی‌های قبلی و هش‌های دقیق ترکیب می‌شوند. گروه‌های دارای چند تصویر حذف نمی‌شوند؛ اعضا در یک بخش قرار می‌گیرند.

In [ ]:
import pandas as pd

# تصاویر مجاز، پس از کنارگذاشتن ۱۹ مورد تعارض
valid_images = set(clean_manifest_v2["image"])

# ترکیب فهرست مقاله با زوج‌هایی که قبلاً خودت بررسی کردی
all_pairs = pd.concat(
    [
        published_pairs[["image_1", "image_2"]],
        reviewed_pairs[["image_1", "image_2"]]
    ],
    ignore_index=True
).drop_duplicates()

# فقط زوج‌هایی را نگه می‌داریم که هر دو تصویرشان باقی مانده‌اند.
usable_pairs = all_pairs[
    all_pairs["image_1"].isin(valid_images)
    & all_pairs["image_2"].isin(valid_images)
].copy()

# ابتدا هر تصویر نمایندهٔ گروه خودش است.
parent = {image: image for image in valid_images}

def find(image):
    """نمایندهٔ نهایی گروه تصویر را پیدا می‌کند."""
    while parent[image] != image:
        parent[image] = parent[parent[image]]
        image = parent[image]
    return image

def union(image_1, image_2):
    """گروه دو تصویر مشابه را به هم وصل می‌کند."""
    root_1 = find(image_1)
    root_2 = find(image_2)

    if root_1 != root_2:
        smaller, larger = sorted([root_1, root_2])
        parent[larger] = smaller

# اتصال تصاویر بر اساس زوج‌های مشابه
for row in usable_pairs.itertuples(index=False):
    union(row.image_1, row.image_2)

# اتصال تکرارهای دقیق ثبت‌شده در بررسی اولیه، در صورت وجود
for _, subset in clean_manifest_v2.dropna(
    subset=["image_hash"]
).groupby("image_hash"):
    images = subset["image"].tolist()
    for image in images[1:]:
        union(images[0], image)

# ساخت نام ثابت برای گروه‌ها
roots = sorted({find(image) for image in valid_images})
group_names = {
    root: f"group_{number:04d}"
    for number, root in enumerate(roots)
}

grouped_manifest_v2 = clean_manifest_v2.copy()
grouped_manifest_v2["group_id"] = (
    grouped_manifest_v2["image"]
    .map(lambda image: group_names[find(image)])
)

# پس از حذف تعارض‌ها، هر گروه باید فقط یک کلاس داشته باشد.
classes_per_group = (
    grouped_manifest_v2.groupby("group_id")["class_name"]
    .nunique()
)
assert classes_per_group.max() == 1, (
    "هنوز یک گروه با برچسب‌های متناقض وجود دارد."
)
assert len(grouped_manifest_v2) == 761

group_sizes = grouped_manifest_v2.groupby("group_id").size()

grouped_manifest_v2.to_csv(
    report_dir_v2 / "grouped_manifest.csv", index=False
)
usable_pairs.to_csv(
    report_dir_v2 / "used_duplicate_pairs.csv", index=False
)

print("Images:", len(grouped_manifest_v2))
print("Groups:", len(group_sizes))
print("Groups with multiple images:", int((group_sizes > 1).sum()))
print("Largest group:", int(group_sizes.max()))
assert len(group_sizes) == 582, 'Grouping differs from the reviewed result.'


## ۹. تقسیم ثابت و ذخیره
هدف در سطح گروه‌ها: ۶۴٪ آموزش، ۱۶٪ اعتبارسنجی، ۲۰٪ آزمون. نتیجه با تقسیم قبلی مقایسه می‌شود. آزمون برای تنظیم مدل استفاده نمی‌شود.

In [ ]:
from sklearn.model_selection import train_test_split
import json

SEED = 42

# یک ردیف برای هر گروه؛ هر گروه فقط یک کلاس دارد.
group_table = (
    grouped_manifest_v2[["group_id", "class_name"]]
    .drop_duplicates()
    .sort_values("group_id")
    .reset_index(drop=True)
)

assert group_table["group_id"].is_unique

# ابتدا ۲۰٪ گروه‌ها را برای آزمون جدا می‌کنیم.
train_val_groups, test_groups = train_test_split(
    group_table,
    test_size=0.20,
    stratify=group_table["class_name"],
    random_state=SEED
)

# سپس ۲۰٪ بخش باقی‌مانده برای اعتبارسنجی:
# ۰٫۲۰ × ۰٫۸۰ = ۰٫۱۶ کل گروه‌ها
train_groups, val_groups = train_test_split(
    train_val_groups,
    test_size=0.20,
    stratify=train_val_groups["class_name"],
    random_state=SEED
)

# تعیین بخش مربوط به هر گروه
split_by_group = {}

for split_name, table in [
    ("train", train_groups),
    ("val", val_groups),
    ("test", test_groups)
]:
    for group_id in table["group_id"]:
        split_by_group[group_id] = split_name

# هر تصویر، بخش گروه خودش را می‌گیرد.
split_manifest_v2 = grouped_manifest_v2.copy()
split_manifest_v2["split"] = (
    split_manifest_v2["group_id"].map(split_by_group)
)

# بررسی کامل‌بودن تقسیم
assert len(split_manifest_v2) == 761
assert split_manifest_v2["image"].is_unique
assert split_manifest_v2["split"].notna().all()

# هیچ گروهی نباید در بیش از یک بخش حضور داشته باشد.
assert (
    split_manifest_v2.groupby("group_id")["split"]
    .nunique()
    .max()
) == 1

# جدول تعداد تصاویر در هر بخش و کلاس
counts = pd.crosstab(
    split_manifest_v2["split"],
    split_manifest_v2["class_name"]
).reindex(["train", "val", "test"])

counts["total"] = counts.sum(axis=1)
display(counts)

print("Image percentages:")
print((100 * counts["total"] / 761).round(2))

# عضویت تصاویر باید با تقسیم تأییدشدهٔ قبلی یکسان باشد.
if reference_split is not None:
    reference = reference_split.set_index('image')['split'].sort_index()
    rebuilt = split_manifest_v2.set_index('image')['split'].sort_index()
    assert reference.equals(rebuilt), 'Split differs from saved result; investigate.'
    print('Membership matches the previously saved split.')
assert counts.loc[['train','val','test'], ['benign','malignant','normal']].values.tolist() == [[281,127,91],[64,35,16],[82,41,24]]

# ذخیرهٔ تقسیم جدید
split_path_v2 = report_dir_v2 / "split_manifest.csv"
split_manifest_v2.to_csv(split_path_v2, index=False)

settings = {
    "seed": SEED,
    "images": len(split_manifest_v2),
    "groups": len(group_table),
    "target_group_ratios": {
        "train": 0.64,
        "val": 0.16,
        "test": 0.20
    },
    "excluded_cross_class_images": 19,
    "grouping_sources": [
        "Published BUSI review, Table B1",
        "35 explicitly recorded previously reviewed similar pairs",
        "Exact decoded-image hashes"
    ],
    "patient_independence_verified": False
}

with open(
    report_dir_v2 / "split_settings.json",
    "w", encoding="utf-8"
) as file:
    json.dump(settings, file, ensure_ascii=False, indent=2)

print("New split saved:", split_path_v2)

## نتیجه و ادامه
انتظار: ۷۶۱ تصویر، ۵۸۲ گروه، ۴۹۹ آموزش، ۱۱۵ اعتبارسنجی، ۱۴۷ آزمون. تقسیم قبلی تغییر نمی‌کند؛ گزارش بازسازی در پوشهٔ جدا ذخیره می‌شود.

برای آموزش در نوت‌بوک جدا، split_manifest.csv تأییدشده را بخوانید. این نسخه استقلال بیماران را تضمین نمی‌کند و حذف سایر موارد مثل تصاویر زیربغل را اعمال نمی‌کند.

ساختار و نحو کد بررسی شده است؛ اجرای کامل به داده‌های Drive و محیط Colab شما نیاز دارد.